In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import openpyxl
import pulp as pl

# =========================================================
# 0. Basic settings
# =========================================================
PRICE_FILE = Path("Prices.xlsx")
EMISSIONS_FACTOR = 0.05306  # tonne CO2 / MMBtu
START_DATE = pd.to_datetime("2022-03-21").date()
END_DATE = pd.to_datetime("2022-03-27").date()

# =========================================================
# 1. Load input data
# =========================================================
price_electric = pd.read_excel(PRICE_FILE, sheet_name="PRICE_ELECTRIC")
price_gas = pd.read_excel(PRICE_FILE, sheet_name="PRICE_GAS")

wb = openpyxl.load_workbook(PRICE_FILE, data_only=True)
co2_sheet = wb["PRICE_CO2"]
CO2_PRICE = float(list(co2_sheet.values)[0][0])

# Based on your actual workbook columns
electric_date_col = "OPERATING_DATE"
electric_hour_col = "HOUR_ENDING"
electric_price_col = "NP15 ($/MWh)"

gas_date_col = "OPERATING_DATE"
gas_price_col = "PG&E Citygate ($/MMBtu)"

elec = price_electric[[electric_date_col, electric_hour_col, electric_price_col]].copy()
elec.columns = ["OPERATING_DATE", "HOUR_ENDING", "PRICE_ELECTRIC"]

gas = price_gas[[gas_date_col, gas_price_col]].copy()
gas.columns = ["OPERATING_DATE", "PRICE_GAS"]

elec["OPERATING_DATE"] = pd.to_datetime(elec["OPERATING_DATE"]).dt.date
gas["OPERATING_DATE"] = pd.to_datetime(gas["OPERATING_DATE"]).dt.date

elec = elec[(elec["OPERATING_DATE"] >= START_DATE) & (elec["OPERATING_DATE"] <= END_DATE)].copy()
gas = gas[(gas["OPERATING_DATE"] >= START_DATE) & (gas["OPERATING_DATE"] <= END_DATE)].copy()

hourly = (
    elec.merge(gas, on="OPERATING_DATE", how="left")
        .sort_values(["OPERATING_DATE", "HOUR_ENDING"])
        .reset_index(drop=True)
)

if hourly["PRICE_GAS"].isna().any():
    raise ValueError("Some gas prices are missing after merge.")

# =========================================================
# 2. Pseudo-unit data
# =========================================================
# Unit 1 = 1x1 block
# Unit 2 = incremental block from 1x1 to 2x1
units = {
    1: {
        "name": "Unit1_1x1",
        "min_mw": 150.0,
        "max_mw": 340.0,
        "ramp_mw_per_min": 20.0,
        "min_up": 2,
        "min_down": 2,
        "su_dollar": 16500.0,
        "su_fuel": 850.0,
        "vom": 2.5,
        "anchor_mw": np.array([150.0, 0.60 * 340.0, 0.80 * 340.0, 340.0]),
        "anchor_hr": np.array([7.695, 7.358, 7.149, 7.048]),
        "anchor_vom": np.array([2.5, 2.5, 2.5, 2.5]),
    },
    2: {
        "name": "Unit2_incremental",
        "min_mw": 312.0 - 150.0,   # 162
        "max_mw": 610.0 - 340.0,   # 270
        "ramp_mw_per_min": 20.0,
        "min_up": 1,
        "min_down": 1,
        "su_dollar": 7250.0,
        "su_fuel": 220.0,
        "anchor_mw": np.array([
            312.0 - 150.0,   # 162
            488.0 - 272.0,   # 216
            610.0 - 340.0    # 270
        ]),
        "anchor_hr": np.array([
            ((7.121 * 312.0) - (7.695 * 150.0)) / (312.0 - 150.0),
            ((6.839 * 488.0) - (7.149 * 272.0)) / (488.0 - 272.0),
            ((6.762 * 610.0) - (7.048 * 340.0)) / (610.0 - 340.0),
        ]),
        "anchor_vom": np.array([
            ((2.0 * 312.0) - (2.5 * 150.0)) / (312.0 - 150.0),
            ((2.0 * 488.0) - (2.5 * 272.0)) / (488.0 - 272.0),
            ((2.0 * 610.0) - (2.5 * 340.0)) / (610.0 - 340.0),
        ]),
    }
}

# =========================================================
# 3. Build 11-point bid curves for each hour
# =========================================================
T = list(range(len(hourly)))
U = list(units.keys())

point_data = {}
point_keys = []

for t in T:
    gas_price = float(hourly.loc[t, "PRICE_GAS"])

    for u in U:
        unit = units[u]

        bid_mw = np.linspace(unit["min_mw"], unit["max_mw"], 11)
        bid_hr = np.interp(bid_mw, unit["anchor_mw"], unit["anchor_hr"])
        bid_vom = np.interp(bid_mw, unit["anchor_mw"], unit["anchor_vom"])

        for i in range(11):
            mw = float(bid_mw[i])
            hr = float(bid_hr[i])
            vom = float(bid_vom[i])

            fuel_cost = hr * gas_price * mw
            co2_cost = hr * EMISSIONS_FACTOR * CO2_PRICE * mw
            vom_cost = vom * mw
            var_cost = fuel_cost + co2_cost + vom_cost

            key = (u, t, i + 1)
            point_keys.append(key)
            point_data[key] = {
                "mw": mw,
                "fuel_cost": fuel_cost,
                "co2_cost": co2_cost,
                "vom_cost": vom_cost,
                "var_cost": var_cost
            }

# =========================================================
# 4. Build MILP
# =========================================================
model = pl.LpProblem("CCGT_PJM_PseudoUnit_Optimization", pl.LpMaximize)

x = pl.LpVariable.dicts("x", [(u, t) for u in U for t in T], cat="Binary")   # online
y = pl.LpVariable.dicts("y", [(u, t) for u in U for t in T], cat="Binary")   # startup
z = pl.LpVariable.dicts("z", [(u, t) for u in U for t in T], cat="Binary")   # shutdown
lam = pl.LpVariable.dicts("lam", point_keys, lowBound=0, upBound=1, cat="Continuous")

# Initial condition: both pseudo-units start offline
x_init = {1: 0, 2: 0}

# Link lambda weights to unit commitment
for u in U:
    for t in T:
        keys_ut = [k for k in point_keys if k[0] == u and k[1] == t]
        model += pl.lpSum(lam[k] for k in keys_ut) == x[(u, t)]

# Startup / shutdown logic
for u in U:
    for t in T:
        if t == 0:
            model += x[(u, t)] - x_init[u] == y[(u, t)] - z[(u, t)]
        else:
            model += x[(u, t)] - x[(u, t-1)] == y[(u, t)] - z[(u, t)]

# Minimum up time
for u in U:
    mut = int(units[u]["min_up"])
    for t in T:
        if t + mut - 1 < len(T):
            model += pl.lpSum(x[(u, tau)] for tau in range(t, t + mut)) >= mut * y[(u, t)]

# Minimum down time
for u in U:
    mdt = int(units[u]["min_down"])
    for t in T:
        if t + mdt - 1 < len(T):
            model += pl.lpSum(1 - x[(u, tau)] for tau in range(t, t + mdt)) >= mdt * z[(u, t)]

# Ramp-rate consistency
p_ut = {}
fuel_cost_ut = {}
co2_cost_ut = {}
vom_cost_ut = {}
var_cost_ut = {}

for u in U:
    for t in T:
        keys_ut = [k for k in point_keys if k[0] == u and k[1] == t]
        p_ut[(u, t)] = pl.lpSum(point_data[k]["mw"] * lam[k] for k in keys_ut)
        fuel_cost_ut[(u, t)] = pl.lpSum(point_data[k]["fuel_cost"] * lam[k] for k in keys_ut)
        co2_cost_ut[(u, t)] = pl.lpSum(point_data[k]["co2_cost"] * lam[k] for k in keys_ut)
        vom_cost_ut[(u, t)] = pl.lpSum(point_data[k]["vom_cost"] * lam[k] for k in keys_ut)
        var_cost_ut[(u, t)] = pl.lpSum(point_data[k]["var_cost"] * lam[k] for k in keys_ut)

for u in U:
    ramp_limit = 60.0 * units[u]["ramp_mw_per_min"]
    for t in T:
        if t > 0:
            model += p_ut[(u, t)] - p_ut[(u, t-1)] <= ramp_limit
            model += p_ut[(u, t-1)] - p_ut[(u, t)] <= ramp_limit

# Startup costs (include non-fuel startup + startup fuel + startup CO2)
startup_nonfuel_t = {}
startup_fuelcost_t = {}
startup_co2cost_t = {}

for t in T:
    gas_price = float(hourly.loc[t, "PRICE_GAS"])
    startup_nonfuel_t[t] = pl.lpSum(units[u]["su_dollar"] * y[(u, t)] for u in U)
    startup_fuelcost_t[t] = pl.lpSum(units[u]["su_fuel"] * gas_price * y[(u, t)] for u in U)
    startup_co2cost_t[t] = pl.lpSum(units[u]["su_fuel"] * EMISSIONS_FACTOR * CO2_PRICE * y[(u, t)] for u in U)

# Objective
model += pl.lpSum(
    float(hourly.loc[t, "PRICE_ELECTRIC"]) * pl.lpSum(p_ut[(u, t)] for u in U)
    - pl.lpSum(var_cost_ut[(u, t)] for u in U)
    - startup_nonfuel_t[t]
    - startup_fuelcost_t[t]
    - startup_co2cost_t[t]
    for t in T
)

# =========================================================
# 5. Solve
# =========================================================
solver = pl.PULP_CBC_CMD(msg=False)
model.solve(solver)

status = pl.LpStatus[model.status]
print("Solve status:", status)

if status != "Optimal":
    raise ValueError(f"Optimization did not solve to Optimal. Status = {status}")

# =========================================================
# 6. Extract hourly results
# =========================================================
rows = []

for t in T:
    unit1_mw = float(pl.value(p_ut[(1, t)]))
    unit2_mw = float(pl.value(p_ut[(2, t)]))

    rows.append({
        "OPERATING_DATE": hourly.loc[t, "OPERATING_DATE"],
        "HOUR_ENDING": int(hourly.loc[t, "HOUR_ENDING"]),
        "PRICE_ELECTRIC": float(hourly.loc[t, "PRICE_ELECTRIC"]),
        "PRICE_GAS": float(hourly.loc[t, "PRICE_GAS"]),
        "MW_GENERATION_Unit1": unit1_mw,
        "MW_GENERATION_Unit2": unit2_mw,
        "FUEL_COST_Unit1": float(pl.value(fuel_cost_ut[(1, t)])),
        "FUEL_COST_Unit2": float(pl.value(fuel_cost_ut[(2, t)])),
        "CO2_COST_Unit1": float(pl.value(co2_cost_ut[(1, t)])),
        "CO2_COST_Unit2": float(pl.value(co2_cost_ut[(2, t)])),
        "VOM_COST_Unit1": float(pl.value(vom_cost_ut[(1, t)])),
        "VOM_COST_Unit2": float(pl.value(vom_cost_ut[(2, t)])),
        "START_Unit1": int(round(pl.value(y[(1, t)]))),
        "START_Unit2": int(round(pl.value(y[(2, t)]))),
        "STARTUP_NONFUEL_Unit1": units[1]["su_dollar"] * int(round(pl.value(y[(1, t)]))),
        "STARTUP_NONFUEL_Unit2": units[2]["su_dollar"] * int(round(pl.value(y[(2, t)]))),
        "STARTUP_FUELCOST_Unit1": units[1]["su_fuel"] * float(hourly.loc[t, "PRICE_GAS"]) * int(round(pl.value(y[(1, t)]))),
        "STARTUP_FUELCOST_Unit2": units[2]["su_fuel"] * float(hourly.loc[t, "PRICE_GAS"]) * int(round(pl.value(y[(2, t)]))),
        "STARTUP_CO2COST_Unit1": units[1]["su_fuel"] * EMISSIONS_FACTOR * CO2_PRICE * int(round(pl.value(y[(1, t)]))),
        "STARTUP_CO2COST_Unit2": units[2]["su_fuel"] * EMISSIONS_FACTOR * CO2_PRICE * int(round(pl.value(y[(2, t)]))),
    })

ccgt_pjm_results = pd.DataFrame(rows)

numeric_cols = [c for c in ccgt_pjm_results.columns if c not in ["OPERATING_DATE"]]
ccgt_pjm_results[numeric_cols] = ccgt_pjm_results[numeric_cols].round(4)

# =========================================================
# 7. Save CSVs
# =========================================================
ccgt_pjm_results[
    ["OPERATING_DATE", "HOUR_ENDING", "PRICE_ELECTRIC", "PRICE_GAS", "MW_GENERATION_Unit1", "MW_GENERATION_Unit2"]
].to_csv("CCGT_PSEUDO.csv", index=False)

ccgt_pjm_results.to_csv("CCGT_PSEUDO_FULL.csv", index=False)

# =========================================================
# 8. First 12 hours
# =========================================================
first12_pjm = ccgt_pjm_results[
    ["OPERATING_DATE", "HOUR_ENDING", "PRICE_ELECTRIC", "PRICE_GAS", "MW_GENERATION_Unit1", "MW_GENERATION_Unit2"]
].head(12)

print("\nFirst 12 hours:")
print(first12_pjm)

# =========================================================
# 9. Summary metrics for each pseudo-unit
# =========================================================
summary_rows = []

for u in U:
    mw_col = f"MW_GENERATION_Unit{u}"
    fuel_col = f"FUEL_COST_Unit{u}"
    co2_col = f"CO2_COST_Unit{u}"
    vom_col = f"VOM_COST_Unit{u}"
    start_col = f"START_Unit{u}"
    startup_nonfuel_col = f"STARTUP_NONFUEL_Unit{u}"
    startup_fuel_col = f"STARTUP_FUELCOST_Unit{u}"
    startup_co2_col = f"STARTUP_CO2COST_Unit{u}"

    total_revenue = float((ccgt_pjm_results["PRICE_ELECTRIC"] * ccgt_pjm_results[mw_col]).sum())
    fuel_cost = float(ccgt_pjm_results[fuel_col].sum() + ccgt_pjm_results[startup_fuel_col].sum())
    co2_cost = float(ccgt_pjm_results[co2_col].sum() + ccgt_pjm_results[startup_co2_col].sum())
    vom_cost = float(ccgt_pjm_results[vom_col].sum())
    startup_nonfuel = float(ccgt_pjm_results[startup_nonfuel_col].sum())
    total_costs = fuel_cost + co2_cost + vom_cost + startup_nonfuel

    gross_margin = total_revenue - total_costs
    gross_margin_per_kw = gross_margin / (units[u]["max_mw"] * 1000.0)
    capacity_factor = float(ccgt_pjm_results[mw_col].sum()) / (units[u]["max_mw"] * len(ccgt_pjm_results))
    number_of_starts = int(ccgt_pjm_results[start_col].sum())

    summary_rows.append({
        "Unit": f"Unit {u}",
        "Gross Margin ($/kW)": round(gross_margin_per_kw, 4),
        "Capacity Factor (%)": round(capacity_factor * 100, 4),
        "Total Revenue ($)": round(total_revenue, 4),
        "Total Costs ($)": round(total_costs, 4),
        "Fuel Costs ($)": round(fuel_cost, 4),
        "Number of Starts": number_of_starts
    })

task3bv_summary = pd.DataFrame(summary_rows)

print("\nTask 3b(v) summary:")
print(task3bv_summary)

Solve status: Optimal

First 12 hours:
   OPERATING_DATE  HOUR_ENDING  PRICE_ELECTRIC  PRICE_GAS  \
0      2022-03-21            1           45.04       7.05   
1      2022-03-21            2           43.63       7.05   
2      2022-03-21            3           43.43       7.05   
3      2022-03-21            4           43.42       7.05   
4      2022-03-21            5           46.30       7.05   
5      2022-03-21            6           55.50       7.05   
6      2022-03-21            7           76.70       7.05   
7      2022-03-21            8           77.29       7.05   
8      2022-03-21            9           51.19       7.05   
9      2022-03-21           10           46.89       7.05   
10     2022-03-21           11           43.54       7.05   
11     2022-03-21           12           37.18       7.05   

    MW_GENERATION_Unit1  MW_GENERATION_Unit2  
0                   0.0                  0.0  
1                   0.0                  0.0  
2                   0.0   